In [1]:
A   

NameError: name 'A' is not defined

In [ ]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.tag import pos_tag
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import numpy as np

def download_nltk_data(resource):
    try:
        nltk.data.find(resource)
        return True  # Already downloaded
    except LookupError:
        try:
            nltk.download(resource)  # Attempt download
            return True  # Download successful
        except Exception as e:
            print(f"Error downloading NLTK resource '{resource}': {e}")
            return False

            
def preprocess_text(text):
    """Cleans and tokenizes text."""
    text = re.sub(r'[^\w\s]', '', text.lower())  # Remove punctuation and lowercase
    stop_words = set(stopwords.words('english'))
    words = word_tokenize(text)
    filtered_words = [w for w in words if w not in stop_words]
    return " ".join(filtered_words)

def identify_tasks(text):
    """Identifies task sentences based on heuristics."""
    sentences = sent_tokenize(text)
    tasks = []
    for sentence in sentences:
        words = word_tokenize(sentence)
        tagged_words = pos_tag(words)
        if len(words)>0:
            if tagged_words[0][1].startswith('VB'):  # Imperative verb start
                tasks.append(sentence)
            elif any(tag[1] == 'CD' for tag in tagged_words) or any(word in sentence.lower() for word in ["deadline","due","by"]): # Check for deadlines/time indicators
                tasks.append(sentence)
            elif any(tag[1] == 'NNP' for tag in tagged_words) : # Check for proper nouns (names)
                tasks.append(sentence)

    return tasks

def categorize_tasks(tasks):
    """Categorizes tasks using TF-IDF and KMeans clustering."""
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(tasks)

    if len(tasks) > 0: # Handle cases where no tasks identified
        num_clusters = min(3, len(tasks)) # Adjust number of clusters based on task count
        kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10) # Set n_init for KMeans convergence
        kmeans.fit(X)
        labels = kmeans.labels_

        categories = {}
        for i, task in enumerate(tasks):
            category_id = labels[i]
            if category_id not in categories:
                categories[category_id] = []
            categories[category_id].append(task)

        #Improve Category Naming (Basic heuristic)
        category_names = {}
        for category_id in categories:
          representative_index = np.argsort(kmeans.transform(vectorizer.transform(categories[category_id]))[0])[0]
          representative_task = categories[category_id][representative_index]
          first_few_words = " ".join(representative_task.split()[:3]) #Take first 3 words
          category_names[category_id] = first_few_words + " related"

        categorized_tasks = {}
        for category_id, task_list in categories.items():
            categorized_tasks[category_names[category_id]] = task_list

        return categorized_tasks
    else:
      return {}


def extract_details(task):
    """Extracts entity and deadline from a task."""
    entity = "Unknown"
    deadline = "Unknown"
    words = word_tokenize(task)
    tagged_words = pos_tag(words)

    for i, (word, tag) in enumerate(tagged_words):
        if tag == 'NNP':  # Proper noun (potential entity)
            entity = word
        elif any(word in time_indicator for time_indicator in ["am","pm","today","tomorrow","tonight","morning","afternoon","evening"]): # Basic time indicators
            deadline = " ".join(words[i:]) #Everything after the indicator is assumed to be the deadline.

    return entity, deadline

def process_text_and_extract_tasks(text):
  processed_text = preprocess_text(text)
  identified_tasks = identify_tasks(processed_text)
  categorized_tasks = categorize_tasks(identified_tasks)

  final_output = {}
  for category, task_list in categorized_tasks.items():
    final_output[category] = []
    for task in task_list:
      entity, deadline = extract_details(task)
      final_output[category].append({"task": task, "entity": entity, "deadline": deadline})
  return final_output

# Example usage (same as before)
text = "Rahul wakes up early every day. He goes to college in the morning and comes back at 3 pm. At present, Rahul is outside. He has to buy the snacks for all of us. John should review the report by tomorrow. Mary needs to schedule a meeting with the team. The project needs to be completed by Friday."
extracted_tasks = process_text_and_extract_tasks(text)

for category, task_details_list in extracted_tasks.items():
    print(f"Category: {category}")
    for task_details in task_details_list:
        print(f"  Task: {task_details['task']}")
        print(f"    Entity: {task_details['entity']}")
        print(f"    Deadline: {task_details['deadline']}")
    print("-" * 20)

# Validation (same as before)
validation_text = "Alice should submit the proposal by Monday. Bob needs to prepare the presentation. Carol has to book the flight tickets. David will clean the office."
validation_extracted_tasks = process_text_and_extract_tasks(validation_text)

print("\n--- Validation Results ---")
for category, task_details_list in validation_extracted_tasks.items():
    print(f"Category: {category}")
    for task_details in task_details_list:
        print(f"  Task: {task_details['task']}")
        print(f"    Entity: {task_details['entity']}")
        print(f"    Deadline: {task_details['deadline']}")
    print("-" * 20)

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - 'C:\\Users\\kumar/nltk_data'
    - 'c:\\Users\\kumar\\OneDrive\\Desktop\\Project\\AI_Assignment (Vijayi_WFH_Tech_internship)\\venv\\nltk_data'
    - 'c:\\Users\\kumar\\OneDrive\\Desktop\\Project\\AI_Assignment (Vijayi_WFH_Tech_internship)\\venv\\share\\nltk_data'
    - 'c:\\Users\\kumar\\OneDrive\\Desktop\\Project\\AI_Assignment (Vijayi_WFH_Tech_internship)\\venv\\lib\\nltk_data'
    - 'C:\\Users\\kumar\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************
